In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:100% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
div.text_cell_render.rendered_html{font-size:18pt;}
div.text_cell_render.rendered_html{font-size:15pt;}
div.output {font-size:18pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:18pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:18pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

In [ ]:
# dataloader_binary.py 먼저 만들기
# model_binary_resnet.py로 모델 구조
# train_binary.py 작성해서 학습
# 성능 확인 후 → 필요 시 증강 적용 또는 ResNet34/Dropout 확장
# 사용 모델 RetNet18 모델

In [ ]:
# 데이터 로드 
"""
food_binary/train, val, test에서 이미지 불러오기
ImageFolder 기반으로 food, not_food를 자동 라벨링
batch_size, num_workers 등 설정 가능하게 하기
train/val/test DataLoader 반환
"""

In [4]:
import torch
import torchvision
import torchvision.transforms as transforms

In [5]:
print(f"torchvision 버전: {torchvision.__version__}")

torchvision 버전: 0.22.1+cpu


In [9]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os

In [ ]:
# 공통 전처리 함수 설정 
# Gpu에서 빠르게 불러오려면 num_workers=4이상조정 (시스템 사양에 맞춰서 진행)
def get_binary_dataloaders(data_dir, batch_size=32, num_workers=2):
    # 기본 경로 지정(경로 수정시 여기에서 !!) / 매개변수에 기본값이 주어지지 않으면 디폴트 경로로 설정 
    if data_dir in None:
        # 하단 경로 수정 필수!★★★★★★★★★★★★
        data_dir = "C:/Users/Admin/Desktop/food_binary"
    
    transform = transforms.Compose([
        transforms.Resize((224,224)), #ResNet18 기본 입력값
        transforms.ToTensor(),
        transforms.Normalize(
        mean = [0.485, 0.46, 0.406],
        std = [0.229, 0.224, 0.225])
    ])
    
 # ★★★★★★★★★★★★★★★★★★ 경로에 맞게 수정 필요
 # 각 폴더에서 데이터셋 불러오기 (food vs not_food) / 폴더 -> Dataset 객체 만들기 
    """
    ★★★★Dataset = 전체 이미지 데이터를 메모리에 로딩 가능한 형태로 만들기     
    ImageFolder 클래스를 label로 인식 / 학습할 수 있는 형태로 만드는 Dataset 클래스 
    파일경로 조합할 때 슬래시 대신 유지보수나 호환선을 위해 os.path.join()을 씀
    transform은 전처리/변형을 적용하는 부분으로 모든 크기 224,224로 통일 / tensor로 변환 : PIL이미지를 pytorch가 이해할 수 있게 바꿔줌
    transform은 정규화 (모델이 빠르고 안정적으로 학습할 수 있게 평균 0, 분산 1로 맞춤)
    """
    train_dataset=datasets.ImageFolder(os.path.join(data_dir,'train'), transform=transform)
    val_dataset=datasets.ImageFolder(os.path.join(data_dir, 'val'), transform=transform)
    test_dataset=datasets.ImageFolder(os.path.join(data_dir,'test'), transform=transform)
    """
    ★★★★DataLoader = 학습할때 데이터 배치(batch)단위로 나눠주는 기능
    ★★★★train_dataset : 학습할 때 한번에 몇장씩 가져올지 지정 / 이미지 32장을 묶어서 미니배치
    shuffle=True 학습용은 데이터 섞기 /검증,테스트는 x
    num_workers=2 : 데이터 로딩을 백그라운드에서 빠르게 수행하는 쓰레드 수 
    이미지를 직접 불러와 label붙인 객체로 만들기
    """
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    
    print ("Loaded classes:", train_dataset.classes) 
    return trani_loader, val_loader, test_loader